# Multi-dataset (Single-Method) Ablations

This notebook demonstrates the new single-method orchestrator with ablation controls.

- We run LoCalPFN only (single method) with different `k` and `fit_adapter` settings.
- You can similarly run TabPFN only by setting `method='tabpfn'` and optionally supplying `tabpfn_clf_kwargs`.


In [ ]:
# Environment/threading caps to mitigate mixed OpenMP issues
import os, site, sys, torch
os.environ['PYTHONNOUSERSITE'] = '1'
usr = site.getusersitepackages(); sys.path = [p for p in sys.path if p != usr]

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except AttributeError:
    pass
print('Thread caps applied; torch intra-op threads = 1')

In [ ]:
# Locate the repo root so relative paths work regardless of where this notebook is launched
from pathlib import Path

def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

repo_root = _add_repo_root_to_sys_path()
print('Repo root:', repo_root)

In [ ]:
# Imports and common paths
from med3pipe.pipelines import run_multi_dataset
from med3pipe.tabular.localpfn import LocalPFNConfig

config_path = repo_root / 'configs' / 'datasets.yaml'
outputs_base = repo_root / 'notebooks'

print('Config path:', config_path, '| Exists:', config_path.exists())
print('Outputs base:', outputs_base)

## LoCalPFN: single-method run (k=25, fit_adapter=False)

In [ ]:
res_loc_25 = run_multi_dataset(
    config_path=config_path,
    method='localpfn',
    outputs_base_dir=outputs_base,
    # Ablation controls
    local_k=25,
    local_fit_adapter=False,
    # Keep adapter sample size small if you enable it later
    local_adapter_num_queries=200,
)
res_loc_25['summary_df']

## LoCalPFN: ablation (k=50, fit_adapter=True, fewer adapter queries)

In [ ]:
res_loc_50a = run_multi_dataset(
    config_path=config_path,
    method='localpfn',
    outputs_base_dir=outputs_base,
    local_k=50,
    local_fit_adapter=True,
    local_adapter_epochs=8,
    local_adapter_lr=5e-2,
    local_adapter_num_queries=150,
)
res_loc_50a['summary_df']

## Compare ablations

In [ ]:
import pandas as pd
summary_loc = pd.concat([res_loc_25['summary_df'], res_loc_50a['summary_df']], ignore_index=True)
summary_loc


---
### Optional: TabPFN single-method run
You can run TabPFN-only by setting `method='tabpfn'`. Pass classifier kwargs via `tabpfn_clf_kwargs` if desired.


In [ ]:
# Example (kept minimal). Uncomment to run.
# res_tab = run_multi_dataset(
#     config_path=config_path,
#     method='tabpfn',
#     outputs_base_dir=outputs_base,
#     tabpfn_clf_kwargs=None,  # e.g., {'N_ensemble_configurations': 16} if supported
# )
# res_tab['summary_df']
